In [1]:
# SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round, sum as spark_sum

spark = SparkSession.builder \
    .appName("Tugas6-ETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession siap. Versi Spark:", spark.version)

26/09/30 14:31:43 WARN Utils: Your hostname, yan resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/09/30 14:31:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 14:31:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# A. EXTRACT
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Transaksi:", df_trx.count(), "baris")
df_trx.printSchema()

print("Produk:", df_produk.count(), "baris")
df_produk.printSchema()

print("Ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

Transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [3]:
# B. TRANSFORM — Penggabungan
# LEFT JOIN transaksi -> ulasan (tidak semua transaksi punya ulasan)
# INNER JOIN transaksi -> produk (setiap transaksi pasti punya produk valid)
df_etl = df_trx \
    .join(df_ulasan, on="order_id", how="left") \
    .join(df_produk, on="product_id", how="inner") \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print("Jumlah baris setelah join:", df_etl.count())  # harus tetap 5000
df_etl.select("order_id", "nama_produk", "kategori", "unit_terjual", "harga",
              "total_pendapatan", "rating").show(5)

Jumlah baris setelah join: 5000
+--------+-----------+------------+------------+------+----------------+------+
|order_id|nama_produk|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+-----------+------------+------------+------+----------------+------+
|     TX0|  Produk-21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|   Produk-8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|  Produk-25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|   Produk-3|     Fashion|           4| 75000|          300000|     5|
|     TX4|  Produk-19|Rumah Tangga|           6| 75000|          450000|  NULL|
+--------+-----------+------------+------------+------+----------------+------+
only showing top 5 rows



In [4]:
# C. TRANSFORM — Penanganan Data Kosong & Pengayaan
# 1) Buat kolom ada_ulasan SEBELUM na.fill(), selagi null masih terbaca
df_etl = df_etl.withColumn("ada_ulasan", col("rating").isNotNull())

# 2) Isi rating kosong dengan 0
df_etl = df_etl.na.fill(0, subset=["rating"])

print("Rating null tersisa:", df_etl.filter(col("rating").isNull()).count())
df_etl.select("order_id", "rating", "ada_ulasan").show(5)

Rating null tersisa: 0
+--------+------+----------+
|order_id|rating|ada_ulasan|
+--------+------+----------+
|     TX0|     1|      true|
|     TX1|     0|     false|
|     TX2|     2|      true|
|     TX3|     5|      true|
|     TX4|     0|     false|
+--------+------+----------+
only showing top 5 rows



# Alasan singkat mengapa 0
Nilai 0 dipilih karena skala rating yang valid adalah 1-5, sehingga 0 berada di luar rentang dan tidak akan tertukar dengan ulasan asli. Nilai 0 dengan jelas menandakan "belum ada ulasan".

In [5]:
# D. LOAD
!hdfs dfs -mkdir -p /user/yan/tugas6

df_etl.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/yan/tugas6/hasil_etl"
)

print("Data berhasil disimpan ke HDFS.")
!hdfs dfs -ls -R /user/yan/tugas6/hasil_etl

Data berhasil disimpan ke HDFS.
-rw-r--r--   3 yan supergroup          0 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - yan supergroup          0 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 yan supergroup      16363 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/kategori=Elektronik/part-00000-49ea7728-a88c-4901-a721-b84bcc3675b4.c000.snappy.parquet
drwxr-xr-x   - yan supergroup          0 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 yan supergroup      11130 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/kategori=Fashion/part-00000-49ea7728-a88c-4901-a721-b84bcc3675b4.c000.snappy.parquet
drwxr-xr-x   - yan supergroup          0 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 yan supergroup      10388 2026-09-30 14:32 /user/yan/tugas6/hasil_etl/kategori=Kesehatan/part-00000-49ea7728-a88c-4901-a721-b84bcc3675b4.c000.snappy.parquet
drwxr-xr-x   - yan supergroup          0 2026-09

In [6]:
df_final = spark.read.parquet("hdfs://localhost:9000/user/yan/tugas6/hasil_etl")
print("Jumlah baris hasil akhir di HDFS:", df_final.count())

Jumlah baris hasil akhir di HDFS: 5000


In [7]:
# E. Insight Akhir
df_insight = df_final.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(col("ada_ulasan").cast("int")).alias("jumlah_ulasan"),
    spark_round(avg(col("ada_ulasan").cast("int")) * 100, 2).alias("persen_ulasan")
).orderBy("persen_ulasan")

df_insight.show()

+------------+---------------+-------------+-------------+
|    kategori|total_transaksi|jumlah_ulasan|persen_ulasan|
+------------+---------------+-------------+-------------+
|     Makanan|            536|          369|        68.84|
|   Kesehatan|            961|          662|        68.89|
|     Fashion|           1035|          726|        70.14|
|Rumah Tangga|            815|          575|        70.55|
|  Elektronik|           1653|         1168|        70.66|
+------------+---------------+-------------+-------------+



# Interpretasi bisnis
Kategori Makanan memiliki persentase transaksi dengan ulasan paling rendah, yaitu 68,84%. Namun selisihnya dengan kategori lain sangat tipis sekitar 68,8% hingga 70,7%, sehingga perbedaan ini belum tentu bermakna secara bisnis. Bagi tim marketing, ulasan berfungsi sebagai bukti sosial yang memengaruhi keputusan beli pelanggan baru.

In [8]:
# Close SparkSession
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
